### PyTorch Dataset

In [ ]:
from torch.utils.data import Dataset
import pandas as pd

class WaterDataset(Dataset):
    def __init__(self, csv_path):
        super().__init__()
        df=pd.read_csv(csv_path)
        self.data = df.to_numpy()

    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, idx):
        features = self.data[idx, :-1]
        label = self.data[idx, -1]
        return features, label

        

### PyTorch DataLoader

In [9]:
dataset_train = WaterDataset('water_potability\\water_train.csv')

In [10]:
from torch.utils.data import DataLoader

DataLoader_train = DataLoader(
    dataset_train, 
    batch_size=2, 
    shuffle=True
    )

TypeError: object of type 'WaterDataset' has no len()

### PyTorch Model
Sequential model definition:

In [ ]:
net = nn.Sequential(
    nn.Linear(9, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid(),
)

Class-based model definition

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(9, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        x = nn.functional.sigmoid(self.fc3(x))
        return x


net = Net()

#### Exercises:


In [ ]:
class WaterDataset(Dataset):
    def __init__(self, csv_path):
        super().__init__()
        # Load data to pandas DataFrame
        df = pd.read_csv(csv_path)
        # Convert data to a NumPy array and assign to self.data
        self.data = df.to_numpy()
        
    # Implement __len__ to return the number of data samples
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, idx):
        features = self.data[idx, :-1]
        # Assign last data column to label
        label = self.data[idx, -1]
        return features, label

In [12]:
# Create an instance of the WaterDataset
dataset_train = WaterDataset('water_train.csv')

# Create a DataLoader based on dataset_train
dataloader_train = DataLoader(
    dataset_train,
    batch_size=2,
    shuffle=True,
)

# Get a batch of features and labels
features, labels = next(iter(dataloader_train))
print(features, labels)

FileNotFoundError: [Errno 2] No such file or directory: 'water_train.csv'

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Define the three linear layers
        self.fc1 = nn.Linear(9, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        
    def forward(self, x):
        # Pass x through linear layers adding activations
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        x = nn.functional.sigmoid(self.fc3(x))
        return x

### Optimizers, training, and evaluation

- Training loop

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Define loss function and optimizer
# BCEloss is used for binary classification problems
# SGD is a simple optimization algorithm that updates the model parameters 
# based on the gradient of the loss function with respect to the parameters.
criterion = nn.BCELoss()

optimizer = optim.SGD(net.parameters(), lr=0.01)

# iterate over epochs and training batches

for epoch in range(1000):
    for features, labels in dataloader_train:
        # Clear gradients
        optimizer.zero_grad()  # Zero the gradients
        # Forward pass: get model's outputs
        outputs = net(features)
        # Compute loss
        loss = criterion(
            outputs, labels.view(-1, 1)
        )
        # Backward pass: compute gradients
        loss.backward()
        # Optimizer's step: update params
        optimizer.step()



Stochastic Gradient Descent (SGD)

 - `optimizer = optim.SGD(net.parameters(), lr=0.01)`
 - update depends on learning rate
 - Simple and efficient, for basic models
 - Rarely used in practice


Adaptive Gradient (Adagrad)
- `optimizer = optim.Adagrad(net.parameters(net.parameters(), lr=0.01))`
- Adapts learning rate for each parameter
- Good for sparse data
- May decrease the learning rate too fast

Root Mean Square Propagation (RMSprop)
- `optimizer = optim.RMSprop(net.parameters(), lr=0.01)`

- Update for each parameter  based on the size of its previous gradients

Adaptive Moment Estimation (Adam)

- `optimzer = optim.Adam(net.parameters(), lr=0.01)`

- Arguably the most versatile and widely used
- RMSprop + gradient momentum
- Often used as te go-to optimizer



Model evaluation

In [ ]:
from torchmetrics import Accuracy

# Set up accuracy metric for binary classification
acc = Accuracy(task='binary')

 

### Exercise

In [ ]:
import torch.optim as optim

net = Net()

# Define the SGD optimizer
optimizer = optim.SGD(net.parameters(), lr=0.001)

train_model(
    optimizer=optimizer,
    net=net,
    num_epochs=10,
)

In [ ]:
import torch.optim as optim

net = Net()

# Define the RMSprop optimizer
optimizer = optim.RMSprop(net.parameters(), lr=0.001)

train_model(
    optimizer=optimizer,
    net=net,
    num_epochs=10,
)

In [ ]:
import torch.optim as optim

net = Net()

# Define the Adam optimizer
optimizer = optim.Adam(net.parameters(), lr=0.01)

train_model(
    optimizer=optimizer,
    net=net,
    num_epochs=10,
)

In [ ]:
import torch
from torchmetrics import Accuracy

# Set up binary accuracy metric
acc = Accuracy(task='binary')
net.eval()
with torch.no_grad():
    for features, labels in dataloader_test:
        # Get predicted probabilities for test data batch
        outputs = net(features)
        preds = (outputs >= 0.5).float()
        acc(preds, labels.view(-1, 1))

# Compute total test accuracy
test_accuracy = acc.compute()
print(f"Test accuracy: {test_accuracy}")

### Vanishing and exploding gradients

##### Vanishing gradients
- Gradients get smaller and smaller during backward pass
- Earlier layers get small parameter updates
- Model doesn't learn

##### Exploding gradients

- Gradients get bigger and bigger
- Parameter updates are too large
- Training diverges

#### Solution to unstable gradients
1. Proper weights  initialization
2. Good activation
3. Batch normalization

#### Weights Initialization
Good initialization ensures:

- Variance of layer inputs = Variance of layer outputs
- Variance of gradients the same before and after a layer

How to achieve this depends on the activation:
- For ReLU and similar, we can use He/Kaiming initialization

In [ ]:
layers = nn.Linear(8,1 )
print(layer.weight)

In [13]:
# Weight Initialization
import torch.nn.init as init 

init.kaiming_uniform_(layer.weight)

print(layer.weight)

NameError: name 'layer' is not defined


KeyboardInterrupt



In [ ]:
# He/Kaiming initialization
init.kaiming_uniform_(self.fc1.weight)
init.kaiming_uniform_(self.fc2.weight)
init.kaiming_uniform_(
    self.fc3.weight,
    nonlinearity="sigmoid",
)

In a full model

In [ ]:
import torch.nn as nn
import torch.nn.init as init

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(9, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

        init.kaiming_uniform_(self.fc1.weight)
        init.kaiming_uniform_(self.fc2.weight)
        init.kaiming_uniform_(
            self.fc3.weight,
            nonlinearity="sigmoid",
        )

### Activation functions
ReLU
- Often used as the default activation
- `nn.functional.relu()`
- Zero for negative inputs - dying neurons


Exponential Linear Unit(ELU) Activation Function
- `nn.functional.elu()`
- Non-zero gradients for negative values - helps against dying neurons
- Average otput around zero - helps against vanishing gradients


#### Batch Normalization
After a layer:
1. Normalization the layer's outputs by:
   - Subtracting the mean
   - Dividing by standard deviation

2. Scale and shift normalized output using learned parameters
Model learns optimal inputs distribution for each layer:

- Faster loss decrease
- Helps against  unstable gradients

In [ ]:
# Batch normalization

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(9, 16)
        self.bn1 = nn.BatchNorm1d(16)
        self.fc2 = nn.Linear(16, 8)
        self.bn2 = nn.BatchNorm1d(8)


    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = nn.functional.elu(x)
        

#### Vanishing gradients
- Gradients get smaller and smaller during backward pass
- Earlier layer get small parameter update
- Model doesn't learn


#### Exploding gradients
- Gradients get bigger and bigger
- Parameter updates are too large
- Training diverge

#### Solution to unstable gradients

1. Proper weights initialization
2. Good Activations

Good initialization ensures:
- Variance of layer inputs = variance of layer outputs
- Variance of gradients the same before and after a layer 


How to achieve this depends on activation:
- For ReLU and similar, we can use HE/Kaiming initialization


In [ ]:
# Weights initialization

layer = nn.Linear(8, 1)
print(layer.weight)

In [ ]:
import torch.nn.init as init

init.kaiming_uniform_(layer.weight)
print(layer.weight)

In [ ]:
# He/Kaiming initialization
init.kaiming_uniform_(self.fc1.weight)
init.kaiming_uniform_(self.fc2.weight)
init.kaiming_uniform_(
    self.fc3.weight,
    nonlinearity="sigmoid",
)

In [ ]:
# Full Model
import torch.nn as nn
import torch.nn.init as init

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(9, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

        init.kaiming_uniform_(self.fc1.weight)
        init.kaiming_uniform_(self.fc2.weight)
        init.kaiming_uniform_(
            self.fc3.weight,
            nonlinearity="sigmoid",
        )

    def forward(self, x):
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        x = nn.functional.sigmoid(self.fc3(x))
        return x

#### Exercise:

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(9, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        
        # Apply He initialization
        init.kaiming_uniform_(self.fc1.weight)
        init.kaiming_uniform_(self.fc2.weight)
        init.kaiming_uniform_(self.fc3.weight, nonlinearity="sigmoid",)

    def forward(self, x):
        # Update ReLU activation to ELU
        x = nn.functional.elu(self.fc1(x))
        x = nn.functional.elu(self.fc2(x))
        x = nn.functional.sigmoid(self.fc3(x))
        return x

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(9, 16)
        # Add two batch normalization layers
        self.bn1 = nn.BatchNorm1d(16)
        self.fc2 = nn.Linear(16, 8)
        self.bn2 = nn.BatchNorm1d(8)
        self.fc3 = nn.Linear(8, 1)
        
        init.kaiming_uniform_(self.fc1.weight)
        init.kaiming_uniform_(self.fc2.weight)
        init.kaiming_uniform_(self.fc3.weight, nonlinearity="sigmoid")
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = nn.functional.elu(x)

        # Pass x through the second set of layers
        x = self.fc2(x)
        x = self.bn2(x)
        x = nn.functional.elu(x)

        x = nn.functional.sigmoid(self.fc3(x))
        return x